1.wconcept > best > 크롤링 [어떤 값을 크롤링 해와야 할 것인가? 고민]
2.크롤링 데이터 MySQL 전송
3.MySQL 전송된 데이터를 가지고 다음과 같은 결과를 출력 (SQL 문법 활용)

-판매가 기준 상위 20개 상품 조회
-할인율이 30% 이상인 상품 조회 & 내림차순 정렬
-특정 브랜드 (임의) 상품만 조회, 판매가 기준 오름차순 정렬
-리뷰수가 100이상인 상품의 평균 판매가 조회
-브랜드별 상품수 & 평균할인률 구하고 상품수가 많은 브랜드 10개 조회

크롤링할 정보:
브랜드명, 판매가격, 상품명, 할인율, 리뷰수

In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import re

URL = "https://display.wconcept.co.kr/"

service = Service(ChromeDriverManager().install())
options = Options()

# options.add_argument("--headless")
# options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--start-maximized")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")
options.add_argument("--no-sandbox")

driver = webdriver.Chrome(service=service, options=options)
driver.get(URL)
time.sleep(2)

best_btn = driver.find_elements(By.CSS_SELECTOR, " button.sc-d57aa1fb-0.iXIzbJ.btn.area-click")
best_btn[1].click()
time.sleep(2)

product_data = list()
products = driver.find_elements(By.CSS_SELECTOR,"div span.sc-228671f3-6.cWZeRh.area-info")
for product in enumerate(products[:50]) :
    try:
        brand = product.find_element(By.CSS_SELECTOR, "span.sc-5ee6d09d-0.hfpiuD.text.title").text.strip()
        product_name = product.find_element(By.CSS_SELECTOR, "span.sc-5ee6d09d-0.hfpiuD.text.detail").text.strip()
        product_price = product.find_element(By.CSS_SELECTOR, "span.sc-5ee6d09d-0.hfpiuD.text.final-price").text.strip()
        dis_rate = product.find_element(By.CSS_SELECTOR, "span.sc-5ee6d09d-0.hfpiuD.text.final-discount").text.strip()
        reviews_count= product.find_element(By.CSS_SELECTOR, "span.sc-5ee6d09d-0.hfpiuD.text.cnt").text.strip()
      
        product_data.append({
            "brand": brand,
            "product_name": product_name,
            "product_price": product_price,
            "dis_rate": dis_rate,
            "reviews_count": reviews_count
        })
    except: 
        dis_rate = ""
        reviews_count = ""
print(product_data)



    
driver.quit()

[{'brand': '시야쥬', 'product_name': '[한정특가] SIOT4079 페이크 레더 봄버 자켓_Black', 'product_price': '123,530', 'dis_rate': '34%', 'reviews_count': '(1,157)'}, {'brand': '메무아', 'product_name': '[20%쿠폰] [모찌언니 PICK] 스터닝 링사 부츠컷 데님', 'product_price': '50,730', 'dis_rate': '43%', 'reviews_count': '(40)'}, {'brand': '올리브데올리브', 'product_name': '오버핏 스웨이드 블루종 점퍼 OE6XJ8740', 'product_price': '174,715', 'dis_rate': '46%', 'reviews_count': '(10)'}, {'brand': '레테라', 'product_name': '[30%+5%더블쿠폰] [BLUE 단독] Loose Fit Stripe Pullover Knit_2color', 'product_price': '71,820', 'dis_rate': '33%', 'reviews_count': '(13)'}, {'brand': '루에브르', 'product_name': 'Wool Hourglass Jacket SW4WJ919-24', 'product_price': '195,426', 'dis_rate': '34%', 'reviews_count': '(403)'}, {'brand': '쥬시쥬디', 'product_name': '[단독] 스웨이드테일러드자켓 JYJK729B', 'product_price': '93,138', 'dis_rate': '47%', 'reviews_count': '(701)'}, {'brand': '온앤온', 'product_name': '스웨이드 봄버 점퍼 NEL6XM801_95', 'product_price': '242,592', 'dis_rate': '39%', 'reviews_count'

In [13]:
import pymysql

MYSQL_HOST = "127.0.0.1"
MYSQL_PORT = 3306
MYSQL_USER = "root"
MYSQL_PW = "Hayeon123!"
MYSQL_DB = "wconcept_crawling"
MYSQL_CHARSET = "utf8mb4"

rows = list()

for el in product_data :
    brand = el.get("brand")
    product_name = el.get("product_name")
    product_price = int(el.get("product_price").replace(",","").strip())
    dis_rate = int(el.get("dis_rate").replace("%", "").strip())
    reviews_count = int(el.get("reviews_count").replace("(","").replace(")","").replace(",","").strip())

    rows.append((brand,product_name,product_price,dis_rate,reviews_count))

print("DB INSERT 대상 row 수:", len(rows))

conn = pymysql.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password=MYSQL_PW,
    database=MYSQL_DB,
    charset=MYSQL_CHARSET
)
#-----여기서부터해--- mysql에서 테이블 생성하는 것 부터 ㄱㄱ
try : 
    with conn.cursor() as cur :
        sql="""
        INSERT INTO wconcept_data (brand, product_name, product_price, dis_rate, reviews_count) 
        VALUES (%s, %s, %s,%s, %s);
        """

        for row in rows : 
            cur.execute(sql, row)
    
    conn.commit()
    print("MySQL 저장완료!")
    
except Exception as e :
    conn.rollback()
    print("저장실패: ", e)

finally :
    conn.close()


DB INSERT 대상 row 수: 43
MySQL 저장완료!
